# 00 - Load Data

In [ ]:
import pandas as pd
import numpy as np

## Cargar datos crudos

In [ ]:
# --- Primer dataset (cronologico de contenido) ---
online_retail_ii = pd.read_excel(r"../data/raw/online_retail_ii.xlsx")

online_retail_ii.rename(columns={
    "Price" : "UnitPrice",
    "Customer ID" : "CustomerID"
}, inplace=True)

print(online_retail_ii.shape)
online_retail_ii.head(5)

In [ ]:
# --- Segundo dataset (cronologico de contenido) ---
online_retail = pd.read_excel(r"../data/raw/online_retail.xlsx")

online_retail.rename(columns={
    "InvoiceNo" : "Invoice"
}, inplace=True)

print(online_retail.shape)
online_retail.head(5)

## Combinar y limpiar

In [ ]:
# Se omiten las ordenes de compra que ya estan en el primer dataset
online_retail = online_retail[~online_retail["Invoice"].isin(online_retail_ii["Invoice"].unique())]

# --- Union de datasets ---
df_raw = pd.concat([
    online_retail_ii,
    online_retail
])
del online_retail_ii, online_retail

# Ordenamiento de dataset
df_raw.sort_values(by=["InvoiceDate","Invoice","StockCode"], inplace=True)

cols_str = ["Invoice","StockCode","Description","Country"]
for col in cols_str:
    df_raw[col] = df_raw[col].astype(str)

cols_float = ["Quantity","UnitPrice","CustomerID"]
for col in cols_float:
    df_raw[col] = df_raw[col].astype(float)

df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])

print(df_raw.shape)
df_raw.head()

In [ ]:
df = df_raw.copy()
del df_raw

df.dropna(subset="CustomerID", inplace=True)
df.sort_values(by=["CustomerID","InvoiceDate","Invoice","StockCode"], inplace=True)
df.reset_index(drop=True, inplace=True)

df["Sales"] = df["Quantity"] * df["UnitPrice"]

print(df.shape)
df.head()

## Construir dataset a nivel cliente

In [ ]:
df_prod = df.copy()

clientes_de_pruebas = df_prod[df_prod["StockCode"].str.casefold().str.contains('test')]["CustomerID"].drop_duplicates()
df_prod = df_prod[~df_prod["CustomerID"].isin(clientes_de_pruebas)]

df_prod_compras = df_prod[df_prod["Quantity"]>=0]

df_prod_devolucion = df_prod[df_prod["Quantity"]<0]

# -- Comrpas reales --
df_customer = df_prod_compras.groupby(by=["CustomerID"]).agg({
    "Country" : pd.Series.nunique,
    "InvoiceDate" : ["min","max"],
    "Sales" : "sum",
    "Quantity" : "sum",
    "Invoice" : pd.Series.nunique,
    "UnitPrice" : "max",
    "StockCode" : pd.Series.nunique
})
df_customer.columns = ["Paises Distintos","Date_Min","Date_Max",  "Sales","Quantity","Compras","Precio_Max","Productos Distintos"]
df_customer.reset_index(inplace=True)


df_customer_top_country = df_prod_compras.groupby(by=["CustomerID","Country"])["Invoice"].nunique().rename("Compras").reset_index().sort_values(by=["CustomerID","Compras"], ascending=False)

df_customer_top_country["rn"] = df_customer_top_country.groupby(by="CustomerID")["CustomerID"].cumcount()
df_customer_top_country = df_customer_top_country[df_customer_top_country["rn"]==0].drop(columns=["rn","Compras"]).rename(columns={"Country":"Pais Principal"})

df_customer = pd.merge(
    df_customer,
    df_customer_top_country,
    on="CustomerID",
    how="left"
)

df_customer.set_index("CustomerID", inplace=True)

df_customer["Permanencia"] = np.round(( df_customer["Date_Max"] - df_customer["Date_Min"] ).dt.total_seconds()/60/60/24, 0) + 1

df_customer["Canasta_Prom"] = df_customer["Quantity"]/df_customer["Compras"]
df_customer["Ticket_Prom"] = df_customer["Sales"]/df_customer["Compras"]
df_customer["Precio_Prom"] = df_customer["Sales"]/df_customer["Quantity"]


# -- Devoluciones --

df_customer_devolucion = df_prod_devolucion.groupby(by="CustomerID")["Invoice"].nunique().rename("Devoluciones").reset_index()


# -- Union de datos --

df_customer = pd.merge(
    df_customer,
    df_customer_devolucion,
    on="CustomerID",
    how="left"
)
df_customer.loc[df_customer["Devoluciones"].isna(), "Devoluciones"] = 0

df_customer["Pct_Devoluciones"] = df_customer["Devoluciones"] / df_customer["Compras"]

df_customer["Pais Principal"] = df_customer["Pais Principal"].astype("category")
df_customer["Paises Distintos"] = df_customer["Paises Distintos"].astype("category")

df_customer = df_customer.set_index("CustomerID")[[
    "Pais Principal", "Paises Distintos",
    "Permanencia",
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos",
    "Pct_Devoluciones"
]]

print(df_customer.shape)
df_customer.head()

## Guardar dataset procesado

In [ ]:
df_customer.to_csv("../data/processed/customer_features.csv", encoding="utf-8")